# Orders by priority (scheduled notebook)

Run by the Airflow DAG `lab_notebook` with papermill, as the batch service identity `lab-batch`: prototype in a notebook, then schedule it (ADR-009). It reads `lakehouse.samples.orders` through Trino and writes a small summary table, `lakehouse.analytics.orders_by_priority`. The executed copy of each run (with its outputs) is kept by Airflow.

The DAG passes a fresh `lab-batch` access token in `LAB_BATCH_TOKEN`; this notebook never prints it.

In [ ]:
# Parameters (papermill replaces this cell's values)
run_id = "manual"
target = "lakehouse.analytics.orders_by_priority"

In [ ]:
import os

import trino

token = os.environ["LAB_BATCH_TOKEN"]
conn = trino.dbapi.connect(
    host=os.environ["LAB_TRINO_HOST"], port=int(os.environ["LAB_TRINO_PORT"]), http_scheme="https",
    verify=os.environ["SSL_CERT_FILE"], auth=trino.auth.JWTAuthentication(token),
    user=os.environ["LAB_BATCH_PRINCIPAL"], catalog="lakehouse", schema="analytics")
cur = conn.cursor()
cur.execute("SELECT current_user")
print("Trino user:", cur.fetchone()[0])

In [ ]:
cur.execute("""
    SELECT orderpriority, count(*) AS orders, round(sum(totalprice), 2) AS revenue
    FROM lakehouse.samples.orders GROUP BY orderpriority ORDER BY orderpriority""")
rows = cur.fetchall()
for r in rows:
    print(r)

In [ ]:
assert target.startswith("lakehouse.analytics."), target
run_lit = run_id.replace("'", "''")  # a SQL string literal
cur.execute(f"""
    CREATE OR REPLACE TABLE {target} AS
    SELECT orderpriority, count(*) AS orders, round(sum(totalprice), 2) AS revenue,
           CAST('{run_lit}' AS varchar) AS run_id, current_timestamp(6) AS computed_at
    FROM lakehouse.samples.orders GROUP BY orderpriority""")
cur.fetchall()
cur.execute(f"SELECT count(*) FROM {target}")
print(target, "rows:", cur.fetchone()[0])